# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bander03/FlyRank_Intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
active = df[df["impressions_90d"] >= 100].copy().reset_index(drop=True)
print(f"Active rows (impressions_90d >= 100): {len(active):,}")

# The exact feature vector used from Week 5 onward for the CTR-underperformance lane.
num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "engagement_rate", "scroll_rate"]
cat_feats = ["content_type"]

for f in num_feats:
    active[f] = active[f].fillna(0)
active["content_type"] = active["content_type"].fillna("unknown")

print("\nMissing values after fill:")
print(active[num_feats + cat_feats].isna().sum())
print("\nFeature vector preview:")
active[num_feats + cat_feats].head()

Active rows (impressions_90d >= 100): 22,006

Missing values after fill:
avg_position              0
impressions_90d           0
word_count                0
content_age_days          0
days_since_last_update    0
engagement_rate           0
scroll_rate               0
content_type              0
dtype: int64

Feature vector preview:


,avg_position,impressions_90d,word_count,content_age_days,days_since_last_update,engagement_rate,scroll_rate,content_type
0,10.6,3803,3221.0,187,20,5.88,4.55,keyword article
1,20.3,15320,2481.0,445,25,0.00,10.00,keyword article
2,36.5,12581,3515.0,141,20,0.00,28.57,keyword article
3,6.2,11751,0.0,463,22,1.28,3.45,keyword article
4,44.0,19140,2803.0,263,14,0.00,24.29,keyword article


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
feature_notes = pd.DataFrame([
    {"feature": "avg_position", "meaning": "mean GSC position over the trailing 90d window",
     "missing_handling": "fillna(0) -- 0 is data-dictionary's own code for 'no position data', so 0-fill preserves that meaning rather than inventing a fake position",
     "available_when": "reported alongside impressions in the same GSC export -- knowable at the same moment as everything else in the row"},
    {"feature": "impressions_90d", "meaning": "GSC search impressions, trailing 90 days",
     "missing_handling": "fillna(0) -- every active row already has >=100 by the filter, so this is a no-op safety fill",
     "available_when": "same export moment"},
    {"feature": "word_count", "meaning": "article word count",
     "missing_handling": "fillna(0) -- blank for ~7,699 rows dataset-wide (not measured); 0 is a real 'unmeasured' signal here, not a guessed value, since word_count_tier already has its own blank-handling upstream",
     "available_when": "static content metadata, known at publish time"},
    {"feature": "content_age_days", "meaning": "days since content creation",
     "missing_handling": "fillna(0) -- not expected to be missing in this slice (every row >= 90 days old); fill is a safety net only",
     "available_when": "known at any point after publish"},
    {"feature": "days_since_last_update", "meaning": "days since the content was last updated",
     "missing_handling": "fillna(0) -- 'never' freshness tier has 0 rows in this slice, so no real fill happens",
     "available_when": "known at any point after the last edit"},
    {"feature": "engagement_rate", "meaning": "engaged_sessions_90d / sessions_90d x 100",
     "missing_handling": "fillna(0) -- 0 sessions implies 0 measurable engagement, a real not a guessed value",
     "available_when": "same trailing-90d export moment"},
    {"feature": "scroll_rate", "meaning": "scroll_events_90d / pageviews_90d x 100 (can exceed 100)",
     "missing_handling": "fillna(0) -- blank only when pageviews_90d=0, where 0 scroll is the honest value",
     "available_when": "same trailing-90d export moment"},
    {"feature": "content_type", "meaning": "keyword article / feedly article / comparison article",
     "missing_handling": "fillna('unknown'), then one-hot encoded with handle_unknown='ignore' so an unseen category at inference time doesn't crash the pipeline",
     "available_when": "static metadata, known at publish time"},
])
feature_notes

,feature,meaning,missing_handling,available_when
0,avg_position,mean GSC position over the trailing 90d window,fillna(0) -- 0 is data-dictionary's own code f...,reported alongside impressions in the same GSC...
1,impressions_90d,"GSC search impressions, trailing 90 days",fillna(0) -- every active row already has >=10...,same export moment
2,word_count,article word count,"fillna(0) -- blank for ~7,699 rows dataset-wid...","static content metadata, known at publish time"
3,content_age_days,days since content creation,fillna(0) -- not expected to be missing in thi...,known at any point after publish
4,days_since_last_update,days since the content was last updated,fillna(0) -- 'never' freshness tier has 0 rows...,known at any point after the last edit
5,engagement_rate,engaged_sessions_90d / sessions_90d x 100,fillna(0) -- 0 sessions implies 0 measurable e...,same trailing-90d export moment
6,scroll_rate,scroll_events_90d / pageviews_90d x 100 (can e...,"fillna(0) -- blank only when pageviews_90d=0, ...",same trailing-90d export moment
7,content_type,keyword article / feedly article / comparison ...,"fillna('unknown'), then one-hot encoded with h...","static metadata, known at publish time"


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42

# This lane's label: low_ctr_for_type (see w05_model.ipynb for the full derivation).
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(active, groups=active["client_id"]))
train, test = active.iloc[tr_idx].copy(), active.iloc[te_idx].copy()

train_type_p40 = train.groupby("content_type")["ctr"].quantile(0.40)
global_p40 = train["ctr"].quantile(0.40)
def label_low_ctr(frame):
    thr = frame["content_type"].map(train_type_p40).fillna(global_p40)
    return (frame["ctr"] <= thr).astype(int)
train["label"] = label_low_ctr(train)
test["label"] = label_low_ctr(test)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def make_pre(feats):
    return ColumnTransformer([("num", "passthrough", [f for f in feats if f != "content_type"]),
                               ("cat", OneHotEncoder(handle_unknown="ignore"), ["content_type"])])

def fit_and_score(feats):
    pre = make_pre(feats)
    rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
    rf.fit(train[feats], train["label"])
    proba = rf.predict_proba(test[feats])[:, 1]
    return {f"precision@{k}": round(float(precision_at_k(proba, test["label"].values, k)), 3) for k in (20, 50, 100)}

honest_feats = num_feats + cat_feats
honest_scores = fit_and_score(honest_feats)
print("HONEST feature set:", honest_feats)
print("Scores:", honest_scores)

# --- Attack 1: label-derived feature (ctr defines the label) ---
leaky_feats = honest_feats + ["ctr"]
leaky_scores = fit_and_score(leaky_feats)
print("\nLEAKY feature set (+ctr):", leaky_scores)
print("-> ctr jumps precision to a flat ~1.0 at every K: label-derived leakage confirmed detectable.")

# --- Attack 2: product-flag-shaped feature (this dataset doesn't ship real ones, so simulate
# the exact failure mode by deriving a column directly from the label itself) ---
train["fake_product_flag"] = train["label"]  # a stand-in for a FlyRank-style pre-computed flag
test["fake_product_flag"] = test["label"]
flagged_feats = honest_feats + ["fake_product_flag"]
pre_flag = make_pre(flagged_feats)
rf_flag = Pipeline([("pre", pre_flag), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_flag.fit(train[flagged_feats], train["label"])
flag_proba = rf_flag.predict_proba(test[flagged_feats])[:, 1]
flag_scores = {f"precision@{k}": round(float(precision_at_k(flag_proba, test['label'].values, k)), 3) for k in (20, 50, 100)}
print("\nPRODUCT-FLAG-SHAPED feature set (+fake_product_flag = a decision, not a signal):", flag_scores)
print("-> Perfect 1.0 immediately -- this is exactly why FlyRank's real product flags")
print("   (health_score, needs_ctr_fix, is_quick_win) are never shipped in this dataset.")

# --- Attack 3: future/overlapping window check ---
print("\nFuture-window check: every feature above is a trailing-90d aggregate 'as of export',")
print("the same moment as the label's own ctr value -- there is no separate future window in")
print("this flat file to accidentally leak from (unlike the warehouse's report_date x day grain,")
print("audited separately in w03_data_contract.ipynb).")

HONEST feature set: ['avg_position', 'impressions_90d', 'word_count', 'content_age_days', 'days_since_last_update', 'engagement_rate', 'scroll_rate', 'content_type']
Scores: {'precision@20': 1.0, 'precision@50': 0.9, 'precision@100': 0.83}



LEAKY feature set (+ctr): {'precision@20': 1.0, 'precision@50': 1.0, 'precision@100': 1.0}
-> ctr jumps precision to a flat ~1.0 at every K: label-derived leakage confirmed detectable.



PRODUCT-FLAG-SHAPED feature set (+fake_product_flag = a decision, not a signal): {'precision@20': 1.0, 'precision@50': 1.0, 'precision@100': 1.0}
-> Perfect 1.0 immediately -- this is exactly why FlyRank's real product flags
   (health_score, needs_ctr_fix, is_quick_win) are never shipped in this dataset.

Future-window check: every feature above is a trailing-90d aggregate 'as of export',
the same moment as the label's own ctr value -- there is no separate future window in
this flat file to accidentally leak from (unlike the warehouse's report_date x day grain,
audited separately in w03_data_contract.ipynb).


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Field | Why excluded |
|---|---|
| `ctr` | Defines this lane's label (`low_ctr_for_type`) directly — Attack 1 above confirms including it collapses the model into a lookup table (precision -> ~1.0 at every K). |
| `trend_pct`, `trend_direction` | This dataset's other label-derived trap: `trend_direction` defines `is_declining_label` used by the repo's reference pipeline. Not this lane's label, but excluded on principle — a column that defines *any* label in this dataset is off-limits as a feature everywhere. |
| `content_id`, `client_id` | Pseudonymous identifiers. Used only to group the train/test split (`client_id`) or as a join key (`content_id`) — never as a model input, since an ID carries no real signal and a model "learning" an ID is just memorizing rows. |
| `provider_used`, `model_used` | Which LLM generated the content. Data dictionary marks these explicitly "not a model feature" — including them would make this a study of authorship, not of search performance. |
| `search_volume`, `competition`, `cpc` | Keyword-context fields, blank for ~2,468 rows with no keyword data. Not used in this lane; including them without handling that structured missingness (it follows `content_type`, not randomness) would silently encode content type twice. |
| `health_score`, `needs_ctr_fix`, `is_quick_win` | FlyRank product flags. Not present in this dataset by design (see `skills/flyrank/flyrank-context/SKILL.md`) — Attack 2 above simulates why: a pre-computed decision as a feature gives a trivial, meaningless perfect score. |

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.